## Code Generation (LSTM)

### Data Acquisition and Vocabulary Building

In [1]:
import urllib.request
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
import torch.optim as optim
import random

# Download MULTIPLE PyTorch files to increase dataset size
urls = [
    "https://raw.githubusercontent.com/pytorch/pytorch/main/torch/nn/modules/linear.py",
    "https://raw.githubusercontent.com/pytorch/pytorch/main/torch/nn/modules/conv.py",
    "https://raw.githubusercontent.com/pytorch/pytorch/main/torch/nn/modules/activation.py",
    "https://raw.githubusercontent.com/pytorch/pytorch/main/torch/nn/modules/dropout.py",
]

text_data = ""
print("Downloading datasets...")
for url in urls:
    response = urllib.request.urlopen(url)
    text_data += response.read().decode('utf-8') + "\n\n"

# Extract unique characters to build the vocabulary
chars = tuple(set(text_data))
int2char = dict(enumerate(chars))
char2int = {ch: ii for ii, ch in int2char.items()}

# Convert the entire text into a sequence of integers
encoded = np.array([char2int[ch] for ch in text_data])

print(f"Total characters in dataset: {len(text_data)}")
print(f"Unique characters (Vocabulary size): {len(chars)}")

Total characters in dataset: 166134
Unique characters (Vocabulary size): 94


### Sequence Generation with Step Logic

In [2]:
seq_length = 150
step = 3  # Shift the sliding window by 3 characters to reduce overlap and memory usage

X = []
y = []

# Create input sequences and target sequences (shifted by 1)
for i in range(0, len(encoded) - seq_length, step):
    seq_in = encoded[i:i + seq_length]
    seq_out = encoded[i + 1:i + seq_length + 1]
    X.append(seq_in)
    y.append(seq_out)

# Convert to PyTorch tensors
X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

# Create Dataset and DataLoader for efficient batching
dataset = TensorDataset(X, y)
train_loader = DataLoader(dataset, batch_size=128, shuffle=True, drop_last=True)

print(f"Total sequences created: {len(X)}")

Total sequences created: 55328


/var/folders/zl/8wcnmw_56bn7v_1jn5_y4mrw0000gn/T/ipykernel_48483/1172898295.py:15: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  X = torch.tensor(X, dtype=torch.long)


### The Generative LSTM Architecture

In [3]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=2, dropout=0.3):
        super(CharRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        
        # LSTM with dropout to prevent overfitting on the source code
        self.lstm = nn.LSTM(
            input_size=embed_size, 
            hidden_size=hidden_size, 
            num_layers=num_layers, 
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        # Fully connected layer maps hidden state to vocabulary distribution
        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x, hidden):
        # x shape: (batch_size, seq_length)
        out = self.embedding(x)
        out, hidden = self.lstm(out, hidden)
        
        # Flatten the output for the Linear layer calculations
        out = out.reshape(-1, out.size(2))
        out = self.fc(out)
        
        return out, hidden

### Training Loop with Independent Batch States

In [4]:
# Model Hyperparameters
vocab_size = len(chars)
embed_size = 64
hidden_size = 256
num_layers = 2
epochs = 50
learning_rate = 0.002

torch.manual_seed(42)

# Device configuration (Supports Apple Silicon MPS, CUDA, or CPU)
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))

model = CharRNN(vocab_size, embed_size, hidden_size, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
best_loss = float('inf')
best_state = None

model.train()
for epoch in range(1, epochs+1):
    total_loss = 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Dynamic hidden state initialization based on model properties
        h0 = torch.zeros(model.lstm.num_layers, inputs.size(0), model.lstm.hidden_size).to(device)
        c0 = torch.zeros(model.lstm.num_layers, inputs.size(0), model.lstm.hidden_size).to(device)
        hidden = (h0, c0)
        
        model.zero_grad()
        output, hidden = model(inputs, hidden)
        
        loss = criterion(output, targets.view(-1))
        loss.backward()
        
        # Gradient Clipping to prevent exploding gradients (very important for LSTMs)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    if avg_loss < best_loss:
        best_loss = avg_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    scheduler.step(avg_loss)
        
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch [{epoch:2d}/{epochs}] - Loss: {avg_loss:.4f} '
              f'- LR: {optimizer.param_groups[0]["lr"]:.5f}')

model.load_state_dict(best_state)
torch.save(model.state_dict(), 'best_char_rnn.pth')
print(f"\nBest loss: {best_loss:.4f} -> saved as best_char_rnn.pth")

Epoch [ 1/50] - Loss: 1.3077 - LR: 0.00200
Epoch [ 5/50] - Loss: 0.2972 - LR: 0.00200
Epoch [10/50] - Loss: 0.2625 - LR: 0.00200
Epoch [15/50] - Loss: 0.2598 - LR: 0.00200
Epoch [20/50] - Loss: 0.2123 - LR: 0.00100
Epoch [25/50] - Loss: 0.2075 - LR: 0.00100
Epoch [30/50] - Loss: 0.2050 - LR: 0.00100
Epoch [35/50] - Loss: 0.2033 - LR: 0.00100
Epoch [40/50] - Loss: 0.2031 - LR: 0.00100
Epoch [45/50] - Loss: 0.2027 - LR: 0.00100
Epoch [50/50] - Loss: 0.1783 - LR: 0.00050

Best loss: 0.1783 -> saved as best_char_rnn.pth


### Inference and Code Generation

In [5]:
def generate_code(model, start_seq, predict_len=600, temperature=0.5):
    model.eval()
    chars_generated = [ch for ch in start_seq]
    
    with torch.no_grad():
        # Dynamically initialize hidden state for batch_size = 1
        h = torch.zeros(model.lstm.num_layers, 1, model.lstm.hidden_size).to(device)
        c = torch.zeros(model.lstm.num_layers, 1, model.lstm.hidden_size).to(device)
        hidden = (h, c)
        
        for ch in start_seq:
            char_tensor = torch.tensor([[char2int[ch]]]).to(device)
            output, hidden = model(char_tensor, hidden)
            
        input_char = torch.tensor([[char2int[start_seq[-1]]]]).to(device)
        
        for _ in range(predict_len):
            output, hidden = model(input_char, hidden)
            
            output = output / temperature
            probabilities = F.softmax(output, dim=1).data
            
            top_char_idx = torch.multinomial(probabilities, 1)[0][0].item()
            chars_generated.append(int2char[top_char_idx])
            
            input_char = torch.tensor([[top_char_idx]]).to(device)
            
    return "".join(chars_generated)

model = CharRNN(vocab_size, embed_size, hidden_size, num_layers).to(device)
state_dict = torch.load('best_char_rnn.pth', weights_only=True)
model.load_state_dict(state_dict)
model.eval()

print("\n--- Generated PyTorch Code ---")
print(generate_code(model, start_seq="class VAE(nn.Module):\n    def __init__(self):",
                    predict_len=250, temperature=0.5))


--- Generated PyTorch Code ---
class VAE(nn.Module):
    def __init__(self):
        return query is an elenet qesed_orgumed_attn mask_and_and
                if input.size(lans (query or degints conting as in_cortacls to expent to entire channels.
    Alpha 1 1
            if fan_input is padding to conspatinal input size a


## Translation (GRU)

### Toy Dataset and Word-Level Vocabulary for Translation

In [6]:
# A tiny synthetic dataset: English to French
pairs = [
    ("i am cold", "j ai froid"),
    ("you are tired", "tu es fatigue"),
    ("he is happy", "il est heureux"),
    ("we are ready", "nous sommes prets"),
    ("they are late", "ils sont en retard")
]

# Define special tokens
SOS_token = 0  # Start Of Sentence
EOS_token = 1  # End Of Sentence
PAD_token = 2  # Padding for batching

def build_vocab(sentences):
    vocab = {"<SOS>": SOS_token, "<EOS>": EOS_token, "<PAD>": PAD_token}
    for sentence in sentences:
        for word in sentence.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab, {v: k for k, v in vocab.items()}

eng_sentences = [p[0] for p in pairs]
fra_sentences = [p[1] for p in pairs]

eng2int, int2eng = build_vocab(eng_sentences)
fra2int, int2fra = build_vocab(fra_sentences)

# Convert sentences to tensor sequences with EOS token
def tensorize(sentence, vocab):
    indexes = [vocab[word] for word in sentence.split()]
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long).view(1, -1) # Shape: (1, seq_len)

print(f"English Vocab Size: {len(eng2int)}")
print(f"French Vocab Size: {len(fra2int)}")

English Vocab Size: 16
French Vocab Size: 19


### The Encoder Architecture

In [7]:
class Encoder(nn.Module):
    def __init__(self, input_size, embed_size, hidden_size):
        super(Encoder, self).__init__()
        self.hidden_size = hidden_size
        
        # Word embedding layer
        self.embedding = nn.Embedding(input_size, embed_size)
        
        # GRU layer (simpler alternative to LSTM, uses only hidden state, no cell state)
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True)
        
    def forward(self, x):
        # x shape: (batch_size, seq_len)
        embedded = self.embedding(x)
        
        # The output contains hidden states for each time step
        # The hidden contains the final state (Context Vector)
        output, hidden = self.gru(embedded)
        
        return hidden

### The Decoder Architecture

In [8]:
class Decoder(nn.Module):
    def __init__(self, output_size, embed_size, hidden_size):
        super(Decoder, self).__init__()
        self.hidden_size = hidden_size
        
        self.embedding = nn.Embedding(output_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True)
        
        # Fully connected layer to predict the next word in the target language
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x, hidden):
        # x shape: (batch_size, 1) - One word at a time
        embedded = self.embedding(x)
        
        output, hidden = self.gru(embedded, hidden)
        
        # Predict the word from the target vocabulary
        prediction = self.fc(output.squeeze(1))
        
        return prediction, hidden

### The Full Seq2Seq Model and Training Logic

In [9]:
# Hyperparameters
hidden_size = 128
embed_size = 64
learning_rate = 0.01
epochs = 200

# Initialize models
encoder = Encoder(len(eng2int), embed_size, hidden_size)
decoder = Decoder(len(fra2int), embed_size, hidden_size)

criterion = nn.CrossEntropyLoss()
encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)

# Training Loop (Batch size = 1 for simplicity in this conceptual example)
encoder.train()
decoder.train()

for epoch in range(epochs):
    total_loss = 0
    
    for eng_text, fra_text in pairs:
        input_tensor = tensorize(eng_text, eng2int)
        target_tensor = tensorize(fra_text, fra2int)
        
        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()
        
        # 1. Pass the input sentence through the Encoder
        encoder_hidden = encoder(input_tensor)
        
        # 2. Prepare Decoder inputs (start with <SOS> and Encoder's final hidden state)
        decoder_input = torch.tensor([[SOS_token]], dtype=torch.long)
        decoder_hidden = encoder_hidden
        
        loss = 0
        target_length = target_tensor.size(1)
        
        # 3. Step through the Decoder
        for t in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            
            # Calculate loss for this target word
            loss += criterion(decoder_output, target_tensor[0, t].unsqueeze(0))
            
            # Teacher Forcing: Feed the actual target word as the next input
            decoder_input = target_tensor[0, t].unsqueeze(0).unsqueeze(0)
            
        loss.backward()
        encoder_optimizer.step()
        decoder_optimizer.step()
        
        total_loss += loss.item() / target_length
        
    if (epoch + 1) % 40 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Average Loss: {total_loss / len(pairs):.4f}")

Epoch [40/200] | Average Loss: 0.0006
Epoch [80/200] | Average Loss: 0.0002
Epoch [120/200] | Average Loss: 0.0001
Epoch [160/200] | Average Loss: 0.0001
Epoch [200/200] | Average Loss: 0.0001


### Inference and Evaluation

In [10]:
def translate_sentence(sentence):
    encoder.eval()
    decoder.eval()
    
    with torch.no_grad():
        input_tensor = tensorize(sentence, eng2int)
        
        # Get Context Vector
        encoder_hidden = encoder(input_tensor)
        
        # Initialize Decoder
        decoder_input = torch.tensor([[SOS_token]], dtype=torch.long)
        decoder_hidden = encoder_hidden
        
        translated_words = []
        
        # Maximum length to prevent infinite loops
        for _ in range(10):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            
            # Get the word with the highest probability
            topv, topi = decoder_output.topk(1)
            predicted_id = topi.squeeze().item()
            
            if predicted_id == EOS_token:
                break
                
            translated_words.append(int2fra[predicted_id])
            
            # The predicted word becomes the next input
            decoder_input = torch.tensor([[predicted_id]], dtype=torch.long)
            
        return " ".join(translated_words)

print("\n--- Translation Test ---")
test_sentences = ["i am cold", "we are ready", "he is happy", "we are cold"]
for sentence in test_sentences:
    print(f"English: {sentence} -> French: {translate_sentence(sentence)}")


--- Translation Test ---
English: i am cold -> French: j ai froid
English: we are ready -> French: nous sommes prets
English: he is happy -> French: il est heureux
English: we are cold -> French: nous sommes prets


That’s why we have Transformers!! XD